In [ ]:
# NORTHSTAR -- Tower 17 Performance: Deep QA for Solace Browser
# DNA: performance(excellent) = speed(sub_second) x memory(bounded) x battery(efficient) x network(minimal) x render(smooth_60fps) x startup(instant) x scale(graceful)
# Extends 07-performance-qa.ipynb with deeper probes on JS/CSS bundle size, asset optimization
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "performance-t17-deep-qa"
NOTEBOOK_PATH = "notebooks/qa/15-performance-t17-qa.ipynb"
TOWER = 17
TOWER_NAME = "Tower of Performance"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"

results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""

print(f"Tower {TOWER}: {TOWER_NAME} -- Deep QA")
assert BROWSER_ROOT.exists()

In [ ]:
# F1 -- Core Web Vitals Auditor (LCP, FID, CLS)
# Q: Are critical CSS and fonts preloaded?
# Q: Does FOUC prevention work?

# Probe: layout.js prevents FOUC with early theme application
layout_js = read_file(WEB / "js" / "layout.js")
has_fouc_prevention = bool(re.search(r'FOUC|theme.*apply|localStorage.*theme|setTheme', layout_js, re.IGNORECASE))
record("F1-P1", "layout.js prevents FOUC", has_fouc_prevention,
       f"{len(layout_js)} bytes")

# Probe: HTML pages preload critical resources
index_html = read_file(WEB / "index.html")
has_preload = bool(re.search(r'rel=["\']preload|rel=["\']preconnect', index_html, re.IGNORECASE))
record("F1-P2", "Index page uses preload/preconnect", has_preload)

# Probe: meta viewport is set for mobile
has_viewport = bool(re.search(r'<meta[^>]*viewport[^>]*width=device-width', index_html, re.IGNORECASE))
record("F1-P3", "Viewport meta tag configured", has_viewport)

# Probe: CSP meta tag present (security + performance)
has_csp = bool(re.search(r'Content-Security-Policy', index_html, re.IGNORECASE))
record("F1-P4", "CSP meta tag present", has_csp)

In [ ]:
# F2 -- JavaScript Performance Engineer
# Q: What is total JS bundle size?
# Q: Are scripts deferred or async?

# Probe: measure total JS size
js_dir = WEB / "js"
js_files = list(js_dir.glob("*.js")) if js_dir.exists() else []
total_js_bytes = sum(f.stat().st_size for f in js_files)
total_js_kb = round(total_js_bytes / 1024, 1)
# Under 500KB total JS is reasonable for a non-SPA
record("F2-P1", "Total JS bundle size under 500KB", total_js_kb < 500,
       f"{total_js_kb}KB across {len(js_files)} files")

# Probe: scripts use defer or async
html_files = list(WEB.glob("*.html"))
scripts_total = 0
scripts_deferred = 0
for hf in html_files:
    content = hf.read_text(encoding='utf-8', errors='replace')
    script_tags = re.findall(r'<script[^>]*src=[^>]*>', content, re.IGNORECASE)
    for tag in script_tags:
        scripts_total += 1
        if re.search(r'defer|async', tag, re.IGNORECASE):
            scripts_deferred += 1
defer_rate = round(scripts_deferred / scripts_total * 100) if scripts_total > 0 else 100
record("F2-P2", "Scripts use defer/async (>50%)", defer_rate >= 50,
       f"{scripts_deferred}/{scripts_total} ({defer_rate}%) scripts deferred/async")

# Probe: no inline scripts longer than 500 chars (render blocking)
long_inline = 0
for hf in html_files:
    content = hf.read_text(encoding='utf-8', errors='replace')
    inline_scripts = re.findall(r'<script(?![^>]*src=)[^>]*>(.*?)</script>', content, re.DOTALL | re.IGNORECASE)
    for script in inline_scripts:
        if len(script.strip()) > 500:
            long_inline += 1
record("F2-P3", "No long inline scripts (>500 chars)", long_inline == 0,
       f"{long_inline} long inline scripts found")

In [ ]:
# F3 -- CSS Performance Specialist
# Q: What is total CSS bundle size?
# Q: Are there unused CSS rules?

# Probe: measure total CSS size
css_files = list((WEB / "css").rglob("*.css")) if (WEB / "css").exists() else []
total_css_bytes = sum(f.stat().st_size for f in css_files)
total_css_kb = round(total_css_bytes / 1024, 1)
record("F3-P1", "Total CSS under 200KB", total_css_kb < 200,
       f"{total_css_kb}KB across {len(css_files)} files")

# Probe: CSS custom properties count is reasonable
site_css = read_file(WEB / "css" / "site.css")
custom_props = set(re.findall(r'--[a-z][a-z0-9-]+', site_css))
record("F3-P2", "CSS custom properties count reasonable (<100)",
       len(custom_props) < 100, f"{len(custom_props)} unique custom properties")

# Probe: no !important overuse
important_count = len(re.findall(r'!important', site_css))
record("F3-P3", "Limited !important usage (<30)", important_count < 30,
       f"{important_count} !important declarations")

# Probe: media queries exist for responsive design
media_queries = re.findall(r'@media\s*\(', site_css)
record("F3-P4", "CSS has responsive media queries", len(media_queries) >= 3,
       f"{len(media_queries)} @media rules")

In [ ]:
# F4 -- Python Server Performance
# Q: Does the server use async where appropriate?
# Q: Are there long-running synchronous operations?

# Probe: web server uses async
server_code = read_file(WEB / "server.py")
has_async = bool(re.search(r'async\s+def|await\s+|aiohttp|asyncio', server_code, re.IGNORECASE))
record("F4-P1", "Web server uses async patterns", has_async,
       f"{len(server_code)} bytes")

# Probe: main server file uses async
main_server = read_file(BROWSER_ROOT / "solace_browser_server.py")
has_main_async = bool(re.search(r'async\s+def|await\s+', main_server))
record("F4-P2", "Main server uses async handlers", has_main_async)

# Probe: no synchronous file reads in request handlers
sync_reads = len(re.findall(r'open\([^)]*\)\.read\(\)', server_code))
record("F4-P3", "No sync file reads in web server handlers", sync_reads == 0,
       f"{sync_reads} sync reads found")

In [ ]:
# F5 -- Image/Asset Optimization + Service Worker
# Q: Does the service worker enable offline caching?
# Q: Are images optimized?

# Probe: service worker exists
sw_path = WEB / "sw.js"
sw_code = read_file(sw_path)
has_sw = bool(re.search(r'cache|fetch|install|activate', sw_code, re.IGNORECASE))
record("F5-P1", "Service worker with caching exists", has_sw,
       f"{len(sw_code)} bytes")

# Probe: manifest.json exists for PWA
manifest = read_file(WEB / "manifest.json")
has_manifest = len(manifest) > 50
record("F5-P2", "Web manifest exists for PWA", has_manifest)

# Probe: images directory exists with reasonable sizes
img_dir = WEB / "images"
if img_dir.exists():
    img_files = list(img_dir.rglob("*"))
    img_files = [f for f in img_files if f.is_file()]
    total_img_kb = round(sum(f.stat().st_size for f in img_files) / 1024, 1)
    large_imgs = [f.name for f in img_files if f.stat().st_size > 500 * 1024]
    record("F5-P3", "No images over 500KB", len(large_imgs) == 0,
           f"{len(large_imgs)} oversized: {large_imgs[:5]}" if large_imgs else f"all under 500KB, total={total_img_kb}KB")
else:
    record("F5-P3", "Images directory exists", False)

In [ ]:
# NEGATIVE SPACE (P16) -- Performance anti-patterns

# Probe: No console.log in production JS
total_console_logs = 0
for js_file in js_files:
    content = js_file.read_text(encoding='utf-8', errors='replace')
    logs = re.findall(r'console\.log\(', content)
    total_console_logs += len(logs)
record("NS-P1", "No excessive console.log in JS (<20)", total_console_logs < 20,
       f"{total_console_logs} console.log calls across {len(js_files)} JS files")

# Probe: robots.txt exists
robots = read_file(WEB / "robots.txt")
has_robots = len(robots) > 10
record("NS-P2", "robots.txt exists", has_robots)

# Probe: sitemap.xml exists
sitemap = read_file(WEB / "sitemap.xml")
has_sitemap = len(sitemap) > 50
record("NS-P3", "sitemap.xml exists", has_sitemap)

# Probe: no duplicate CSS files
css_names = [f.name for f in css_files]
duplicates = [n for n in css_names if css_names.count(n) > 1]
record("NS-P4", "No duplicate CSS file names", len(set(duplicates)) == 0,
       f"Duplicates: {set(duplicates)}" if duplicates else "all unique")

In [ ]:
# EVIDENCE SUMMARY -- Tower 17 Performance Deep QA
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"

summary = {
    "surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_probes": total, "passed": passed, "failed": failed,
    "score": score, "grade": grade, "finding_rate": finding_rate,
    "converged": finding_rate < 5,
    "findings": [r for r in results if r["status"] == "FAIL"],
    "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]
}

print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME} -- DEEP QA")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))